# Robustness check: cross-seed variance (executed run)

The actual run, outputs intact. To rerun it, upload this notebook to Kaggle, do the setup below, then **Run all** (the outputs are overwritten).

Retrains each winning config (MLP and CNN) multiple times with different `training_seed` values, keeping `split_seed` fixed so every run trains/evaluates on the identical split. This measures how much the reported test accuracy varies from random init/shuffling alone, vs. being a single lucky run.

Same Kaggle setup as `kaggle_run_executed.ipynb`/`kaggle_run_executed_cnn.ipynb` — GPU + internet on, `mnist-mlp-pipeline` dataset (latest version, republished after this notebook's changes) and the `digit-recognizer` competition attached. Distribution-shift robustness is *not* run here — that part needs no retraining, so it runs locally against the committed checkpoints instead: `python scripts/shift_robustness.py` (see `REPORT.md` §6.2).

In [1]:
!pip install -q pytorch-lightning torchmetrics optuna python-dotenv torchvision

In [7]:
import sys

DATASET_DIR = '/kaggle/input/datasets/ivannamor/mnist-mlp-pipeline'
sys.path.insert(0, DATASET_DIR)
print('project code ready')

project code ready


In [12]:
from src.config import ExperimentConfig

mlp_config = ExperimentConfig.from_yaml(f'{DATASET_DIR}/configs/best_from_search.yaml')
mlp_config.data.data_dir = '/kaggle/input/competitions/digit-recognizer'

cnn_config = ExperimentConfig.from_yaml(f'{DATASET_DIR}/configs/cnn_best_from_search.yaml')
cnn_config.data.data_dir = '/kaggle/input/competitions/digit-recognizer'

print(mlp_config)
print(" ")
print(cnn_config)

ExperimentConfig(architecture='mlp', data=DataConfig(data_dir='/kaggle/input/competitions/digit-recognizer', train_csv='train.csv', val_fraction=0.2, test_fraction=0.2, batch_size=128, num_workers=2, split_seed=42), model=ModelConfig(hidden_dims=[512, 256], activation='gelu', dropout=0.258885920517189, num_classes=10), optim=OptimConfig(optimizer='adamw', lr=0.0009157770842261716, weight_decay=0.00043489682497903484, label_smoothing=0.1, lr_scheduler=None), trainer=TrainerConfig(max_epochs=30, accelerator='auto', devices=1, patience=5, experiment_name='best_from_search', training_seed=42))
 
ExperimentConfig(architecture='cnn', data=DataConfig(data_dir='/kaggle/input/competitions/digit-recognizer', train_csv='train.csv', val_fraction=0.2, test_fraction=0.2, batch_size=64, num_workers=2, split_seed=42), model=CNNModelConfig(conv_channels=[64, 128], kernel_size=3, activation='gelu', dropout=0.015026644895738628, fc_hidden_dim=256, num_classes=10), optim=OptimConfig(optimizer='adam', lr=0

## Cross-seed trials -- MLP (5 seeds, ~5 full training runs)

In [13]:
from src.robustness import run_cross_seed_trials

SEEDS = [42, 43, 44, 45, 46]
mlp_seed_results = run_cross_seed_trials(mlp_config, SEEDS)
for r in mlp_seed_results:
    print(r)

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net                   │ Sequential                │  535 K │ train │     0 │
│ 1 │ loss_fn               │ CrossEntropyLoss          │      0 │ train │     0 │
│ 2 │ val_accuracy          │ MulticlassAccuracy        │      0 │ train │     0 │
│ 3 │ val_precision         │ MulticlassPrecision       │      0 │ train │     0 │
│ 4 │ val_recall            │ MulticlassRecall          │      0 │ train │     0 │
│ 5 │ test_accuracy         │ MulticlassAccuracy        │      0 │ train │     0 │
│ 6 │ test_precision        │ MulticlassPrecision       │      0 │ train │     0 │
│ 7 │ test_recall           │ MulticlassRecall          │      0 │ train │     0 │
│ 8 │ test_confusion_matrix │ MulticlassConfusionMatrix │      0 │ train │     0 │
└───┴───────────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 535 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 535 K                                                                                                
Total estimated model params size (MB): 2.143                                                                      
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=30` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Seed set to 43
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Ligh

┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net                   │ Sequential                │  535 K │ train │     0 │
│ 1 │ loss_fn               │ CrossEntropyLoss          │      0 │ train │     0 │
│ 2 │ val_accuracy          │ MulticlassAccuracy        │      0 │ train │     0 │
│ 3 │ val_precision         │ MulticlassPrecision       │      0 │ train │     0 │
│ 4 │ val_recall            │ MulticlassRecall          │      0 │ train │     0 │
│ 5 │ test_accuracy         │ MulticlassAccuracy        │      0 │ train │     0 │
│ 6 │ test_precision        │ MulticlassPrecision       │      0 │ train │     0 │
│ 7 │ test_recall           │ MulticlassRecall          │      0 │ train │     0 │
│ 8 │ test_confusion_matrix │ MulticlassConfusionMatrix │      0 │ train │     0 │
└───┴───────────────────────┴───────────────────────────┴────────┴───────┴───────┘

...

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Seed set to 46
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.12/

┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net                   │ Sequential                │  535 K │ train │     0 │
│ 1 │ loss_fn               │ CrossEntropyLoss          │      0 │ train │     0 │
│ 2 │ val_accuracy          │ MulticlassAccuracy        │      0 │ train │     0 │
│ 3 │ val_precision         │ MulticlassPrecision       │      0 │ train │     0 │
│ 4 │ val_recall            │ MulticlassRecall          │      0 │ train │     0 │
│ 5 │ test_accuracy         │ MulticlassAccuracy        │      0 │ train │     0 │
│ 6 │ test_precision        │ MulticlassPrecision       │      0 │ train │     0 │
│ 7 │ test_recall           │ MulticlassRecall          │      0 │ train │     0 │
│ 8 │ test_confusion_matrix │ MulticlassConfusionMatrix │      0 │ train │     0 │
└───┴───────────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 535 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 535 K                                                                                                
Total estimated model params size (MB): 2.143                                                                      
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=30` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


{'training_seed': 42, 'val_accuracy': 0.9786615371704102, 'test_accuracy': 0.9819406270980835}
{'training_seed': 43, 'val_accuracy': 0.979790210723877, 'test_accuracy': 0.98127281665802}
{'training_seed': 44, 'val_accuracy': 0.9795750379562378, 'test_accuracy': 0.9828169345855713}
{'training_seed': 45, 'val_accuracy': 0.9784020185470581, 'test_accuracy': 0.98160320520401}
{'training_seed': 46, 'val_accuracy': 0.9793829917907715, 'test_accuracy': 0.9823376536369324}


## Cross-seed trials -- CNN (5 seeds, ~5 full training runs)

In [14]:
cnn_seed_results = run_cross_seed_trials(cnn_config, SEEDS)
for r in cnn_seed_results:
    print(r)

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /kaggle/working/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net                   │ Sequential                │  1.7 M │ train │     0 │
│ 1 │ loss_fn               │ CrossEntropyLoss          │      0 │ train │     0 │
│ 2 │ val_accuracy          │ MulticlassAccuracy        │      0 │ train │     0 │
│ 3 │ val_precision         │ MulticlassPrecision       │      0 │ train │     0 │
│ 4 │ val_recall            │ MulticlassRecall          │      0 │ train │     0 │
│ 5 │ test_accuracy         │ MulticlassAccuracy        │      0 │ train │     0 │
│ 6 │ test_precision        │ MulticlassPrecision       │      0 │ train │     0 │
│ 7 │ test_recall           │ MulticlassRecall          │      0 │ train │     0 │
│ 8 │ test_confusion_matrix │ MulticlassConfusionMatrix │      0 │ train │     0 │
└───┴───────────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 1.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.7 M                                                                                                
Total estimated model params size (MB): 6.732                                                                      
Modules in train mode: 20                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=30` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Seed set to 43
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Ligh

┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net                   │ Sequential                │  1.7 M │ train │     0 │
│ 1 │ loss_fn               │ CrossEntropyLoss          │      0 │ train │     0 │
│ 2 │ val_accuracy          │ MulticlassAccuracy        │      0 │ train │     0 │
│ 3 │ val_precision         │ MulticlassPrecision       │      0 │ train │     0 │
│ 4 │ val_recall            │ MulticlassRecall          │      0 │ train │     0 │
│ 5 │ test_accuracy         │ MulticlassAccuracy        │      0 │ train │     0 │
│ 6 │ test_precision        │ MulticlassPrecision       │      0 │ train │     0 │
│ 7 │ test_recall           │ MulticlassRecall          │      0 │ train │     0 │
│ 8 │ test_confusion_matrix │ MulticlassConfusionMatrix │      0 │ train │     0 │
└───┴───────────────────────┴───────────────────────────┴────────┴───────┴───────┘

...

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=30` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Seed set to 46
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Ligh

┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net                   │ Sequential                │  1.7 M │ train │     0 │
│ 1 │ loss_fn               │ CrossEntropyLoss          │      0 │ train │     0 │
│ 2 │ val_accuracy          │ MulticlassAccuracy        │      0 │ train │     0 │
│ 3 │ val_precision         │ MulticlassPrecision       │      0 │ train │     0 │
│ 4 │ val_recall            │ MulticlassRecall          │      0 │ train │     0 │
│ 5 │ test_accuracy         │ MulticlassAccuracy        │      0 │ train │     0 │
│ 6 │ test_precision        │ MulticlassPrecision       │      0 │ train │     0 │
│ 7 │ test_recall           │ MulticlassRecall          │      0 │ train │     0 │
│ 8 │ test_confusion_matrix │ MulticlassConfusionMatrix │      0 │ train │     0 │
└───┴───────────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 1.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.7 M                                                                                                
Total estimated model params size (MB): 6.732                                                                      
Modules in train mode: 20                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=30` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


{'training_seed': 42, 'val_accuracy': 0.9921683073043823, 'test_accuracy': 0.9921367168426514}
{'training_seed': 43, 'val_accuracy': 0.9923530220985413, 'test_accuracy': 0.9924755692481995}
{'training_seed': 44, 'val_accuracy': 0.9914193153381348, 'test_accuracy': 0.9921362400054932}
{'training_seed': 45, 'val_accuracy': 0.9925245046615601, 'test_accuracy': 0.9925929307937622}
{'training_seed': 46, 'val_accuracy': 0.9922124147415161, 'test_accuracy': 0.9927177429199219}


## Summary: mean +/- std across seeds

MLP: mean=0.9820 std=0.0006 min=0.9813 max=0.9828 — CNN: mean=0.9924 std=0.0003 min=0.9921 max=0.9927. See `README.md`'s "Robustness checks" section for the full interpretation.

In [6]:
import statistics

def summarize(name, results):
    accs = [r['test_accuracy'] for r in results]
    print(f"{name}: mean={statistics.mean(accs):.4f}  std={statistics.stdev(accs):.4f}  min={min(accs):.4f}  max={max(accs):.4f}")
    return accs

mlp_accs = summarize('MLP', mlp_seed_results)
cnn_accs = summarize('CNN', cnn_seed_results)


MLP: mean=0.9820  std=0.0006  min=0.9813  max=0.9828
CNN: mean=0.9924  std=0.0003  min=0.9921  max=0.9927
